In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
pip install stable-baselines3 gymnasium numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 4.2 MB/s eta 0:00:0000:01
Note: you may need to restart the kernel to use updated packages.


**GREEDY BASELINE**

In [ ]:
# =========================================
# GREEDY BASELINE 
# =========================================

import json
import copy
import numpy as np
import pandas as pd
from math import radians, sin, cos, sqrt, atan2, ceil

# -------------------------
# REUSE ALL CONSTANTS & HELPERS FROM DQN CODE
# (paste your existing constants here, or import them)
# -------------------------

# Paste: MINUTES_PER_STEP, STABILIZATION_TIME, MORTALITY_RATES,
#        transport_survival_rate, haversine_distance,
#        total_free_hospital_beds, total_free_shelter_space,
#        generate_severity_breakdown, DisasterEnv
# ... (all unchanged from your DQN code)


# =========================================
# GREEDY POLICY
# =========================================

def greedy_action(obs, zone_type, ambulances, buses, boats):
    """
    Greedy rule-based policy:
      - Always prefer saving people over doing nothing
      - Prioritize injured (critical/serious) if present
      - Use boats for floods, buses otherwise
      - Do BOTH if resources allow
    """
    critical    = obs[0]
    serious     = obs[1]
    moderate    = obs[2]
    displaced   = obs[3]
    total_beds  = obs[4]
    total_shelter = obs[5]
    
    has_injured  = (critical + serious + moderate) > 0
    has_displaced = displaced > 0
    can_treat    = has_injured and total_beds > 0 and ambulances > 0
    can_bus      = has_displaced and total_shelter > 0 and buses > 0
    can_boat     = has_displaced and total_shelter > 0 and boats > 0 and zone_type == "flood"

    # Both: treat injured + move displaced simultaneously
    if can_treat and (can_bus or can_boat):
        return 4  # BOTH

    # Injured only (critical/serious present, no displaced transport possible)
    if can_treat:
        return 1  # INJURED ONLY

    # Displaced only
    if can_boat:
        return 3  # BOAT (flood priority)
    if can_bus:
        return 2  # BUS

    # Nothing can be done
    return 0  # NOOP


# =========================================
# EVALUATION LOOP
# =========================================

with open("/kaggle/working/resource_database_greedy_82.json") as f:
    data = json.load(f)

ZONES = list(data["zones"].values())
print(f"✅ Loaded {len(ZONES)} zones")

eval_env = DisasterEnv(ZONES, survival_mode=True)
zone_rows = []

for z in ZONES:
    # --- Manual env reset  ---
    eval_env.zone     = copy.deepcopy(z)
    eval_env.clusters = eval_env.zone["people_clusters"]

    for c in eval_env.clusters:
        if "severity" not in c:
            injured_total = c.get("injured_total", c.get("injured", 0))
            c["severity"]        = generate_severity_breakdown(injured_total)
            c["severity_alive"]  = copy.deepcopy(c["severity"])
            c["dead_by_severity"]= {"critical": 0, "serious": 0, "moderate": 0}

    eval_env.hospitals = eval_env.zone["hospitals"]
    eval_env.shelters  = eval_env.zone["shelters"]

    tr = eval_env.zone["transport_resources"]
    eval_env.ambulances        = tr["ambulances"]
    eval_env.ambulance_capacity= tr["ambulance_capacity"]
    eval_env.buses             = tr["buses"]
    eval_env.bus_capacity      = tr["bus_capacity"]

    dr = eval_env.zone["disaster_resources"]
    eval_env.boats        = dr.get("boats", 0)
    eval_env.boat_capacity= dr.get("boat_capacity", 10)

    eval_env.cluster_idx   = 0
    eval_env.timestep      = 0
    eval_env.hospital_queue= []
    eval_env._reset_report()

    obs, _ = eval_env.reset()
    done   = False

    while not done:
        action = greedy_action(
            obs,
            zone_type  = eval_env.zone["disaster_type"],
            ambulances = eval_env.ambulances,
            buses      = eval_env.buses,
            boats      = eval_env.boats
        )
        obs, _, done, _, _ = eval_env.step(action)

    # --- Compute metrics  ---
    total_injured_original = sum(
        c["severity"]["critical"] + c["severity"]["serious"] + c["severity"]["moderate"]
        for c in eval_env.clusters
    )
    injured_still_alive  = sum(
        c["severity_alive"]["critical"] + c["severity_alive"]["serious"] + c["severity_alive"]["moderate"]
        for c in eval_env.clusters
    )
    displaced_still_unsafe = sum(c["uninjured_displaced"] for c in eval_env.clusters)
    total_displaced_original = sum(c["uninjured_displaced"] for c in z["people_clusters"])

    total_dead = (
        eval_env.report["died_critical"] +
        eval_env.report["died_serious"]  +
        eval_env.report["died_moderate"] +
        eval_env.report["died_transport"]
    )

    actual_injured_saved  = total_injured_original - injured_still_alive - total_dead
    actual_displaced_saved= total_displaced_original - displaced_still_unsafe

    total_people = total_injured_original + total_displaced_original
    total_saved  = actual_injured_saved + actual_displaced_saved

    eval_env.report.update({
        "total_injured"      : total_injured_original,
        "total_displaced"    : total_displaced_original,
        "total_people"       : total_people,
        "injured_saved"      : actual_injured_saved,
        "displaced_saved"    : actual_displaced_saved,
        "total_saved"        : total_saved,
        "total_dead"         : total_dead,
        "overall_save_%"     : (total_saved / total_people * 100)   if total_people            > 0 else 0,
        "injured_save_%"     : (actual_injured_saved / total_injured_original * 100) if total_injured_original > 0 else 0,
        "displaced_save_%"   : (actual_displaced_saved / total_displaced_original * 100) if total_displaced_original > 0 else 0,
        "mortality_%"        : (total_dead / total_injured_original * 100) if total_injured_original > 0 else 0,
    })

    zone_rows.append(eval_env.report)

# =========================================
# RESULTS
# =========================================

zone_df = pd.DataFrame(zone_rows)

print("\n" + "="*120)
print("GREEDY BASELINE RESULTS")
print("="*120)
cols = ["zone_id", "disaster_type", "total_people", "total_saved", "overall_save_%",
        "injured_saved", "displaced_saved", "injured_save_%", "displaced_save_%", "mortality_%"]
print(zone_df[cols].to_string(index=False))

print("\n" + "="*120)
print("OVERALL STATISTICS")
print("="*120)
total_inj = zone_df["total_injured"].sum()
total_dis = zone_df["total_displaced"].sum()
total_ppl = zone_df["total_people"].sum()
saved_inj = zone_df["injured_saved"].sum()
saved_dis = zone_df["displaced_saved"].sum()
saved_all = saved_inj + saved_dis
total_dead= zone_df["total_dead"].sum()

print(f"Total injured saved  : {saved_inj:,} / {total_inj:,}  ({saved_inj/total_inj*100:.1f}%)")
print(f"Total displaced saved: {saved_dis:,} / {total_dis:,}  ({saved_dis/total_dis*100:.1f}%)")
print(f"OVERALL SAVE RATE    : {saved_all:,} / {total_ppl:,}  ({saved_all/total_ppl*100:.1f}%)")
print(f"Total deaths         : {total_dead:,}")
print(f"Beds freed (turnover): {zone_df['beds_freed'].sum():,}")

zone_df.to_csv("/kaggle/working/zone_summary_greedy.csv", index=False)
print("\n💾 Saved: zone_summary_greedy.csv")
print(f"\n✅ GREEDY BASELINE COMPLETE!")
print(f"Achieved: {saved_all/total_ppl*100:.1f}%")


OUTCOMES | INJURED BREAKDOWN | DISPLACED | TRANSPORT | BEDS | SHELTER
Zone  Disaster People   Saved    Save %   Inj    Critical  Serious   Moderate  Dis saved  Amb    Bus    Boat   Beds   Shelter Occ 
--------------------------------------------------------------------------------------------------------------
4     Wind     1,111    737      66.34%   275    54        95        126       462        138    10     0      195    462         
12    Wind     10,384   8,535    82.19%   3,295  658       1,152     1,485     5,240      1,648  105    0      2,842  5,240       
14    Wind     6,974    5,481    78.59%   2,474  494       865       1,115     3,007      1,237  61     0      2,120  3,007       
26    Wind     1,465    1,244    84.91%   434    86        150       198       810        217    17     0      314    810         
28    Flood    3,782    3,088    81.65%   992    196       346       450       2,096      496    0      210    852    2,096       
2     Wind     13,901   13,236  